# Build a VXLAN EVPN Knowledge Graph — in Python

The data-center sequel to *Build an OSPF Knowledge Graph*. Same idea, same tools
(**Python + networkx**, zero setup in Colab), but now the graph models a **VXLAN
BGP-EVPN fabric** — the spine-leaf network under your virtual machines and containers.

**The one idea:** an EVPN fabric has *two* planes stacked on top of each other:

- the **underlay** — plain IP reachability between the switches (usually built by OSPF
  or BGP). It answers: *can these two switches ping each other's loopback?*
- the **overlay** — the VXLAN tunnels that carry tenant traffic, advertised by
  **BGP-EVPN**. It answers: *which host lives behind which switch, on which segment?*

The crux, and the whole reason to model it: **the overlay is only as reachable as the
underlay beneath the VTEP.** A tunnel can be perfectly configured and still carry
nothing if the underlay can't reach the far switch's loopback. This notebook makes that
failure a single property you can flip.

**Is this machine learning?** No — same as the OSPF course. Structured facts you can
question, nothing learned or guessed.


## The words you need

- **VTEP / Leaf** — a leaf switch that wraps and unwraps VXLAN traffic. It owns a
  loopback IP (its `vtep_ip`) that all tunnels point at. *Node label:* `Leaf`.
- **Spine** — the fabric core every leaf connects to. Carries the **underlay** only;
  it never sees tenant traffic unwrapped. *Node label:* `Spine`.
- **VNI** — VXLAN Network Identifier, the 24-bit segment number. An **L2 VNI** is one
  broadcast domain (a stretched VLAN); an **L3 VNI** is a tenant routing table (a VRF).
  *Node label:* `VNI`.
- **Host** — an endpoint (VM / container) identified by its MAC+IP. It sits **on** a VNI
  and **behind** exactly one leaf. *Node label:* `Host`.
- **Service / Tenant** — the business thing that runs on a segment, inside a tenant.
  *Node label:* `Service`.
- **underlay** — IP reachability between VTEPs. We store it as a `Leaf --UNDERLAY-->
  Spine` edge carrying a `state` of `up` or `down`.
- **overlay** — the VXLAN/EVPN relationships: which leaf **HOSTS** which VNI, which host
  is **ON** a VNI and **BEHIND** a leaf, which service **RUNS_ON** a VNI.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# networkx and matplotlib come pre-installed in Colab.
# A DiGraph is a *directed* graph: every edge has a direction, like our arrows.
G = nx.DiGraph()
print('empty fabric graph ready:', G)

## Step 1 — Create the fabric nodes

One spine, three leaves (each a VTEP with its own loopback), one stretched L2 segment,
two hosts on it, and the business service that rides the segment. Every node gets a
**label** (its type) and **properties** (facts about it).


In [ ]:
# the fabric core
G.add_node('spine-1', label='Spine')

# three leaf switches — each is a VTEP with a loopback (vtep_ip)
G.add_node('leaf-1', label='Leaf', vtep_ip='10.0.0.1')
G.add_node('leaf-2', label='Leaf', vtep_ip='10.0.0.2')
G.add_node('leaf-3', label='Leaf', vtep_ip='10.0.0.3')

# an L2 segment (stretched VLAN) and an L3 segment (tenant VRF), both tenant 'Blue'
G.add_node('vni-10010', label='VNI', kind='L2', vni=10010, tenant='Blue')
G.add_node('vni-50000', label='VNI', kind='L3', vni=50000, tenant='Blue')

# two hosts (VM / container endpoints) that live on the L2 segment
G.add_node('host-A', label='Host', mac='02:00:00:00:0a:01', ip='10.10.10.11')
G.add_node('host-B', label='Host', mac='02:00:00:00:0a:02', ip='10.10.10.12')

# the business service running on the segment, inside tenant Blue
G.add_node('Payments API', label='Service', tenant='Blue', criticality='critical')

print(G.number_of_nodes(), 'nodes:', list(G.nodes))

## Step 2 — Wire the two planes

**Underlay first.** Each leaf reaches the spine by plain IP. That reachability is the
foundation every tunnel stands on, so we put its `state` right on the edge. `leaf-1`'s
underlay is **down** — its loopback `10.0.0.1` is now unreachable across the fabric.

**Overlay second.** `HOSTS` = a leaf participates in a VNI. `ON` / `BEHIND` place a host
on a segment and behind a leaf. `RUNS_ON` ties the service to its segment.


In [ ]:
# --- UNDERLAY: IP reachability from each leaf to the spine ---
G.add_edge('leaf-1', 'spine-1', rel='UNDERLAY', state='down')   # <-- the failure
G.add_edge('leaf-2', 'spine-1', rel='UNDERLAY', state='up')
G.add_edge('leaf-3', 'spine-1', rel='UNDERLAY', state='up')

# --- OVERLAY: which leaf participates in which VNI ---
G.add_edge('leaf-1', 'vni-10010', rel='HOSTS')
G.add_edge('leaf-2', 'vni-10010', rel='HOSTS')
G.add_edge('leaf-1', 'vni-50000', rel='HOSTS')
G.add_edge('leaf-2', 'vni-50000', rel='HOSTS')

# --- OVERLAY: hosts sit ON a segment and BEHIND a leaf ---
G.add_edge('host-A', 'vni-10010', rel='ON')
G.add_edge('host-A', 'leaf-1',    rel='BEHIND')   # host-A lives behind the broken leaf
G.add_edge('host-B', 'vni-10010', rel='ON')
G.add_edge('host-B', 'leaf-2',    rel='BEHIND')

# --- OVERLAY: the service runs on the L2 segment ---
G.add_edge('Payments API', 'vni-10010', rel='RUNS_ON')

for u, v, d in G.edges(data=True):
    tag = f"({d['state']})" if 'state' in d else ''
    print(f'{u:14} --{d["rel"]}{tag}--> {v}')

## See it — the two planes in one picture

Colour the nodes by label. The **down** underlay edge is drawn red and dashed: that is
the single fact that decides whether the overlay above it means anything.


In [ ]:
colors = {'Spine': '#0b5d55', 'Leaf': '#0f7f74', 'VNI': '#3aa0ff',
          'Host': '#9aa5ad', 'Service': '#c8702a'}
node_colors = [colors.get(G.nodes[n].get('label'), '#cccccc') for n in G]
pos = nx.spring_layout(G, seed=7)   # seed keeps the layout stable

plt.figure(figsize=(10, 7))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1700, alpha=0.92)
nx.draw_networkx_labels(G, pos, font_size=8)

down  = [(u, v) for u, v, d in G.edges(data=True) if d.get('state') == 'down']
other = [(u, v) for u, v, d in G.edges(data=True) if (u, v) not in down]
nx.draw_networkx_edges(G, pos, edgelist=other, arrows=True, arrowsize=15, edge_color='#8894a0')
nx.draw_networkx_edges(G, pos, edgelist=down,  arrows=True, arrowsize=15, edge_color='#d34b3f', style='dashed', width=2)
nx.draw_networkx_edge_labels(G, pos, font_size=6.5,
    edge_labels={(u, v): d['rel'] for u, v, d in G.edges(data=True)})

plt.axis('off'); plt.title('VXLAN EVPN knowledge graph (underlay + overlay)')
plt.tight_layout(); plt.show()

## Step 3 — Ask it the 3 AM question

*When a leaf's underlay reachability goes down — its VTEP is unreachable in the fabric —
which VNIs, tenants and services, and which hosts, lose reachability?*

We **traverse**: a leaf whose `UNDERLAY` edge is `down` -> the VNIs it `HOSTS` ->
the services that `RUNS_ON` those VNIs, and the hosts sitting `BEHIND` that leaf.
Those hosts go dark to the rest of the fabric because remote VTEPs can no longer reach
this leaf's loopback to deliver VXLAN traffic.


In [ ]:
def blast_radius(G):
    """What is at risk when a leaf's underlay (VTEP reachability) is down."""
    vnis_at_risk, services_at_risk, hosts_at_risk = set(), set(), set()
    for leaf, spine, d in G.edges(data=True):
        # 1) a leaf whose UNDERLAY edge to the fabric is down
        if d.get('rel') != 'UNDERLAY' or d.get('state') != 'down':
            continue
        vtep = G.nodes[leaf].get('vtep_ip')
        # 2) the VNIs that leaf participates in (HOSTS)
        for _, vni, d2 in G.out_edges(leaf, data=True):
            if d2.get('rel') != 'HOSTS':
                continue
            vnis_at_risk.add(vni)
            # 3) services that run on those VNIs (RUNS_ON, edge points INTO the vni)
            for svc, _, d3 in G.in_edges(vni, data=True):
                if d3.get('rel') == 'RUNS_ON':
                    services_at_risk.add(svc)
        # 4) hosts sitting BEHIND that leaf (edge points INTO the leaf)
        for host, _, d4 in G.in_edges(leaf, data=True):
            if d4.get('rel') == 'BEHIND':
                hosts_at_risk.add(host)
        print(f'leaf {leaf} (vtep {vtep}) underlay DOWN')
    return vnis_at_risk, services_at_risk, hosts_at_risk

vnis, services, hosts = blast_radius(G)
print('  VNIs unreachable  :', sorted(vnis))
print('  services at risk  :', sorted(services))
print('  hosts cut off     :', sorted(hosts))

The switch's own logs would only tell you an underlay adjacency dropped. Your graph
just told you the **Payments API** is at risk and **host-A** is cut off — because you
modelled the overlay riding on top of the underlay. That is the EVPN lesson in one
answer: **the overlay is only as reachable as the underlay beneath the VTEP.**


## What-if: repair the underlay, then break it again

Flip `leaf-1`'s underlay to `up` and ask again — nothing is at risk, because now the
far VTEPs *can* reach its loopback and the tunnels carry traffic. Set it back to `down`
— the Payments API and host-A return. Same overlay config the whole time; only the
underlay `state` changed. That is the toggle worth internalising.


In [ ]:
G['leaf-1']['spine-1']['state'] = 'up'      # repair the underlay
v, s, h = blast_radius(G)
print('  after repair -> services at risk:', sorted(s) or 'nothing at risk')
print()

G['leaf-1']['spine-1']['state'] = 'down'    # break it again
v, s, h = blast_radius(G)
print('  after re-break -> services at risk:', sorted(s), '| hosts cut off:', sorted(h))

## Make it yours

Change the values below to **one** leaf, **one** VNI and **one** service *you* run, then
re-run. Your own service and host come back from `blast_radius(G)` — the moment the tool
is yours. Keep the shape; change only the values. (Model the smallest slice that answers
one real question — you can always add one more leaf or VNI later.)


In [ ]:
# --- change these values to your fabric ---
MY_LEAF    = 'leaf-dc2-07'
MY_VTEP    = '10.0.2.7'
MY_VNI     = 'vni-30020'
MY_HOST    = 'db-primary-01'
MY_SERVICE = 'Orders DB'
MY_TENANT  = 'Green'
# ------------------------------------------

G.add_node(MY_LEAF,    label='Leaf', vtep_ip=MY_VTEP)
G.add_node(MY_VNI,     label='VNI',  kind='L2', tenant=MY_TENANT)
G.add_node(MY_HOST,    label='Host')
G.add_node(MY_SERVICE, label='Service', tenant=MY_TENANT, criticality='critical')

G.add_edge(MY_LEAF, 'spine-1', rel='UNDERLAY', state='down')   # the modelled failure
G.add_edge(MY_LEAF, MY_VNI,    rel='HOSTS')
G.add_edge(MY_HOST, MY_VNI,    rel='ON')
G.add_edge(MY_HOST, MY_LEAF,   rel='BEHIND')
G.add_edge(MY_SERVICE, MY_VNI, rel='RUNS_ON')

vnis, services, hosts = blast_radius(G)
print('services at risk:', sorted(services))
print('hosts cut off   :', sorted(hosts))

## You built a VXLAN EVPN knowledge graph

From an empty graph to a question no switch CLI answers: *which tenants and hosts lose
reachability when one VTEP falls off the underlay?* The shapes you used —
*a service runs on a VNI, a leaf hosts a VNI, a host sits behind a leaf, a leaf's
underlay reaches the fabric* — are the whole model.

**The lesson to keep:** the overlay is only as reachable as the underlay beneath the
VTEP. A flawless BGP-EVPN config buys you nothing if the underlay can't reach the far
loopback — and now you have a graph that says so out loud.

**Where next:** the companion OSPF lab builds the *underlay* half in its own right, and
the full course does this exact model in **Neo4j + Cypher** — same nodes, same edges,
same blast-radius question.
